In [1]:
# Install required packages
!pip install -q pymupdf pandas pillow

import os
import subprocess

# Repository URL
repo_url = "https://github.com/Anukriti12/PDF-Accessibility-Benchmark.git"

# Download repository
repo_dir = "/content/PDF-Accessibility-Benchmark"

if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", repo_url, repo_dir],
        check=True
    )

print("Benchmark repository downloaded!")

Benchmark repository downloaded!


In [2]:
# Locate alternative text docs
import os

data_dir = os.path.join(repo_dir, "data", "inputs")

print("Available criterion folders:")

for folder in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder)

    if os.path.isdir(folder_path):
        print(folder)

Available criterion folders:
color_contrast
semantic_tagging
alt_text_quality
logical_reading_order
fonts_readability
table_structure
functional_hyperlinks


In [3]:
# Inspect alternative text docs
# Find the Alternative Text Quality folder
alt_text_dir = None

for root, dirs, files in os.walk(data_dir):
    if os.path.basename(root).lower() == "alt_text_quality":
        alt_text_dir = root
        break

if alt_text_dir is None:
    raise FileNotFoundError(
        "Could not locate Alternative Text Quality folder. "
        "Check the criterion folder names above."
    )

print("Alternative Text directory:", alt_text_dir)

# Inspect benchmark labels and files
for label in os.listdir(alt_text_dir):

    label_path = os.path.join(alt_text_dir, label)

    if os.path.isdir(label_path):

        print(f"\nLABEL: {label}")

        for root, dirs, files in os.walk(label_path):

            for filename in files[:]:
                print(
                    os.path.relpath(
                        os.path.join(root, filename),
                        alt_text_dir
                    )
                )

Alternative Text directory: /content/PDF-Accessibility-Benchmark/data/inputs/alt_text_quality

LABEL: cannot_tell
cannot_tell/.DS_Store
cannot_tell/W3005755974/W3005755974_0.html
cannot_tell/W3005755974/W3005755974_0.pdf
cannot_tell/W3005755974/inputs/W3005755974_0#0.jpg
cannot_tell/W3005755974/inputs/W3005755974_0.png
cannot_tell/W4383621582/W4383621582_0.pdf
cannot_tell/W4383621582/W4383621582_0.html
cannot_tell/W4383621582/inputs/W4383621582_0#0.jpg
cannot_tell/W4383621582/inputs/W4383621582_0.png
cannot_tell/W2929611936/W2929611936_0.pdf
cannot_tell/W2929611936/W2929611936_1.html
cannot_tell/W2929611936/W2929611936_0.html
cannot_tell/W2929611936/W2929611936_1.pdf
cannot_tell/W2929611936/inputs/W2929611936_0.png
cannot_tell/W2929611936/inputs/W2929611936_1#1.jpg
cannot_tell/W2929611936/inputs/W2929611936_1#0.jpg
cannot_tell/W2929611936/inputs/W2929611936_0#0.jpg
cannot_tell/W2929611936/inputs/W2929611936_1.png
cannot_tell/W2460269320/W2460269320_1.html
cannot_tell/W2460269320/W24602

In [4]:
import os
import pandas as pd

# Creating inventory of files
benchmark_files = []

for root, dirs, files in os.walk(alt_text_dir):

    for filename in files:

        file_path = os.path.join(root, filename)

        extension = os.path.splitext(filename)[1].lower()

        relative_path = os.path.relpath(
            file_path,
            alt_text_dir
        )

        benchmark_files.append({
            "filename": filename,
            "extension": extension,
            "file_path": file_path,
            "relative_path": relative_path
        })

benchmark_inventory = pd.DataFrame(benchmark_files)

print("Total benchmark files:", len(benchmark_inventory))

print("\nFile distribution:")
print(benchmark_inventory["extension"].value_counts())

display(benchmark_inventory.head(20))

Total benchmark files: 139

File distribution:
extension
.html    28
.pdf     28
.png     28
.jpg     28
.txt     21
          6
Name: count, dtype: int64


,filename,extension,file_path,relative_path
0,.DS_Store,,/content/PDF-Accessibility-Benchmark/data/inpu...,.DS_Store
1,.DS_Store,,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/.DS_Store
2,W3005755974_0.html,.html,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W3005755974/W3005755974_0.html
3,W3005755974_0.pdf,.pdf,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W3005755974/W3005755974_0.pdf
4,W3005755974_0#0.jpg,.jpg,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W3005755974/inputs/W3005755974_0#0...
5,W3005755974_0.png,.png,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W3005755974/inputs/W3005755974_0.png
6,W4383621582_0.pdf,.pdf,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W4383621582/W4383621582_0.pdf
7,W4383621582_0.html,.html,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W4383621582/W4383621582_0.html
8,W4383621582_0#0.jpg,.jpg,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W4383621582/inputs/W4383621582_0#0...
9,W4383621582_0.png,.png,/content/PDF-Accessibility-Benchmark/data/inpu...,cannot_tell/W4383621582/inputs/W4383621582_0.png


In [7]:

import os
import pandas as pd

# Building benchmark dataframe

# Benchmark label mapping
label_mapping = {
    "passed": 1,
    "failed": 0,
    "not_present": 0
}

benchmark_records = []

# Image formats
image_extensions = {".jpg"}

for root, dirs, files in os.walk(alt_text_dir):

    # Determine benchmark label from directory structure
    relative_dir = os.path.relpath(root, alt_text_dir)
    label_folder = relative_dir.split(os.sep)[0].lower()

    # Exclude Cannot Tell and any unrecognized folders
    if label_folder not in label_mapping:
        continue

    benchmark_label = label_mapping[label_folder]

    # Index text files by filename stem
    text_files = {
        os.path.splitext(f)[0]: os.path.join(root, f)
        for f in files
        if f.lower().endswith(".txt")
    }

    # Locate images
    for filename in files:

        extension = os.path.splitext(filename)[1].lower()

        if extension not in image_extensions:
            continue

        image_path = os.path.join(root, filename)
        image_stem = os.path.splitext(filename)[0]

        # Find matching text file
        text_path = text_files.get(image_stem)

        # Read matching text file
        if text_path and os.path.exists(text_path):

            with open(
                text_path,
                "r",
                encoding="utf-8",
                errors="replace"
            ) as f:

                alt_text = f.read().strip()

        else:
            alt_text = ""

        # Identify document
        document_id = image_stem.split("#")[0]

        benchmark_records.append({

            "document_id": document_id,

            "page_id": None,

            "figure_id": image_stem,

            "image_path": image_path,

            "alt_text": alt_text,

            "alt_text_class": benchmark_label,

            "alt_text_type": (
                "meaningful"
                if benchmark_label == 1
                else "nonmeaningful"
            ),

            "dataset_source": "PDF Accessibility Benchmark",

            "label_source": "document_level_benchmark",

            "benchmark_original_label": label_folder,

            "label_granularity": "document",

            "manual_review_required": True

        })


# Create dataframe
benchmark_df = pd.DataFrame(benchmark_records)

print("Benchmark extraction complete!")

print("Total candidate records:", len(benchmark_df))

if len(benchmark_df) > 0:

    print("\nOriginal benchmark label distribution:")
    print(benchmark_df["benchmark_original_label"].value_counts())

    print("\nBinary classification distribution:")
    print(benchmark_df["alt_text_type"].value_counts())

    print("\nAlt text availability:")
    print(
        (benchmark_df["alt_text"].str.strip() != "").value_counts()
    )

    display(benchmark_df.head())


Benchmark extraction complete!
Total candidate records: 21

Original benchmark label distribution:
benchmark_original_label
failed         7
passed         7
not_present    7
Name: count, dtype: int64

Binary classification distribution:
alt_text_type
nonmeaningful    14
meaningful        7
Name: count, dtype: int64

Alt text availability:
alt_text
True     14
False     7
Name: count, dtype: int64


,document_id,page_id,figure_id,image_path,alt_text,alt_text_class,alt_text_type,dataset_source,label_source,benchmark_original_label,label_granularity,manual_review_required
0,W3005755974_0,None,W3005755974_0#0,/content/PDF-Accessibility-Benchmark/data/inpu...,Figure 6,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
1,W4383621582_0,None,W4383621582_0#0,/content/PDF-Accessibility-Benchmark/data/inpu...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
2,W2929611936_1,None,W2929611936_1#1,/content/PDF-Accessibility-Benchmark/data/inpu...,Embedded Image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
3,W2929611936_1,None,W2929611936_1#0,/content/PDF-Accessibility-Benchmark/data/inpu...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
4,W2929611936_0,None,W2929611936_0#0,/content/PDF-Accessibility-Benchmark/data/inpu...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True


In [8]:
import os
import shutil
import pandas as pd
from google.colab import drive

# =====================================================
# STEP 1: Mount Google Drive
# =====================================================

drive.mount('/content/drive')

# =====================================================
# STEP 2: Create benchmark_data_folder
# =====================================================

output_dir = (
    "/content/drive/MyDrive/EEE6778_Semester_Project/"
    "benchmark_data_folder"
)

image_output_dir = os.path.join(output_dir, "images")

os.makedirs(image_output_dir, exist_ok=True)

# =====================================================
# STEP 3: Verify benchmark dataframe
# =====================================================

if benchmark_df.empty:
    raise ValueError("benchmark_df is empty. Check the extraction steps.")

# Create a copy so the original dataframe remains unchanged
final_df = benchmark_df.copy()

# =====================================================
# STEP 4: Copy images into the images folder
# =====================================================

new_image_paths = []
copied_images = 0
missing_images = []

for index, row in final_df.iterrows():

    original_path = row["image_path"]

    if not os.path.exists(original_path):
        new_image_paths.append(None)
        missing_images.append(original_path)
        continue

    # Preserve original filename
    filename = os.path.basename(original_path)

    # Destination path
    destination_path = os.path.join(
        image_output_dir,
        filename
    )

    # Copy image
    shutil.copy2(original_path, destination_path)

    # Store FULL Google Drive path
    new_image_paths.append(destination_path)

    copied_images += 1

# Update dataframe with full image paths
final_df["image_path"] = new_image_paths

print("Images copied:", copied_images)
print("Missing images:", len(missing_images))

print("\nExample image paths:")
for path in final_df["image_path"].dropna().head(5):
    print(path)


# =====================================================
# STEP 5: Save dataset as CSV
# =====================================================

csv_path = os.path.join(
    output_dir,
    "pdf_benchmark_dataset.csv"
)

final_df.to_csv(csv_path, index=False)

# =====================================================
# STEP 6: Save dataset as Excel
# =====================================================

excel_path = os.path.join(
    output_dir,
    "pdf_benchmark_dataset.xlsx"
)

final_df.to_excel(excel_path, index=False)

# =====================================================
# STEP 7: Display results
# =====================================================

print("BENCHMARK DATASET SAVED SUCCESSFULLY")
print("-------------------------------------")

print("Output folder:", output_dir)

print("Total dataset records:", len(final_df))

print("Images successfully copied:", copied_images)

print("Missing images:", len(missing_images))

print("CSV saved:", os.path.exists(csv_path))

print("Excel saved:", os.path.exists(excel_path))

print("Images folder exists:", os.path.exists(image_output_dir))

print("\nFinal dataset preview:")
display(final_df.head())

if missing_images:
    print("\nMissing image files:")
    for path in missing_images[:10]:
        print(path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Images copied: 21
Missing images: 0

Example image paths:
/content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder/images/W3005755974_0#0.jpg
/content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder/images/W4383621582_0#0.jpg
/content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder/images/W2929611936_1#1.jpg
/content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder/images/W2929611936_1#0.jpg
/content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder/images/W2929611936_0#0.jpg
BENCHMARK DATASET SAVED SUCCESSFULLY
-------------------------------------
Output folder: /content/drive/MyDrive/EEE6778_Semester_Project/benchmark_data_folder
Total dataset records: 21
Images successfully copied: 21
Missing images: 0
CSV saved: True
Excel saved: True
Images folder exists: True

Final dataset preview:


,document_id,page_id,figure_id,image_path,alt_text,alt_text_class,alt_text_type,dataset_source,label_source,benchmark_original_label,label_granularity,manual_review_required
0,W3005755974_0,None,W3005755974_0#0,/content/drive/MyDrive/EEE6778_Semester_Projec...,Figure 6,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
1,W4383621582_0,None,W4383621582_0#0,/content/drive/MyDrive/EEE6778_Semester_Projec...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
2,W2929611936_1,None,W2929611936_1#1,/content/drive/MyDrive/EEE6778_Semester_Projec...,Embedded Image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
3,W2929611936_1,None,W2929611936_1#0,/content/drive/MyDrive/EEE6778_Semester_Projec...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
4,W2929611936_0,None,W2929611936_0#0,/content/drive/MyDrive/EEE6778_Semester_Projec...,image,0,nonmeaningful,PDF Accessibility Benchmark,document_level_benchmark,failed,document,True
